In [2]:
import torch
from torch.utils.data import DataLoader, TensorDataset, random_split
from collections import OrderedDict
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import scanpy as sc
import numpy as np
import pandas as pd
from torchsummary import summary
from torchviz import make_dot
from tqdm import tqdm
import csv
from scipy.sparse import issparse
from model.sparsemax import Sparsemax

from model.dataset import BagsDataset, custom_collate_fn

In [3]:
adata = sc.read_h5ad('/project/DPDS/Wang_lab/shared/spatial_TCR/data/train_validate/Visium/HumanOvarianCancer/T_cell.h5ad')
dataset = BagsDataset(adata, immune_cell='tcell', radius=150, n_genes=500, resolution='low')
train_size = int(0.7 * len(dataset))
val_size = len(dataset) - train_size
train_dataset, val_dataset = random_split(dataset, [train_size, val_size])
#dataset = load_datasets(data_dir='/project/DPDS/Wang_lab/s439765/spatial_tcr/HumanLungCancerFFPE', immune_cell='tcell',radius=200,)

Tumor cells shape after filtering: (1226, 17943)
Selecting top 500 genes based on mean expression
Percentile value: 4.037618160247803
adata.obs[T] after binarization: AAACAAGTATCTCCCA-1    0
AAACAATCTACTAGCA-1    0
AAACACCAATAACTGC-1    1
AAACAGAGCGACTCCT-1    1
AAACAGCTTTCAGAAG-1    0
Name: T, dtype: int64
Preprocessed data: (3455, 500)


Creating Bags with radius 150: 100%|█████████████████████████| 3455/3455 [00:00<00:00, 22320.30it/s]

Total bags created: 1226
Average instances per bag: 4


In [4]:
dataloader = torch.utils.data.DataLoader(train_dataset, batch_size=1, shuffle=True, collate_fn=custom_collate_fn)
val_dataloader = torch.utils.data.DataLoader(val_dataset, batch_size=1, shuffle=True, collate_fn=custom_collate_fn)

distances, gene_expressions, labels, core_index, current_genes = next(iter(dataloader))


In [49]:

class Distance(nn.Module):
    def __init__(self):
        super(Distance, self).__init__()
        self.a = nn.Parameter(torch.tensor(-3.0),requires_grad=True)
        #self.sparsemax = Sparsemax(dim=0)
        self.softmax = nn.Softmax(dim=0)

    
    def forward(self, x):
        #print(x)
        a = self.a
        x = self.softmax(-torch.exp(a) * x)
        return x


In [50]:
model = Distance()
output = model(distances[0])
print(output)

tensor([[8.7224e-04],
        [8.7224e-04],
        [8.7224e-04],
        [9.9651e-01],
        [8.7224e-04]], grad_fn=<SoftmaxBackward0>)


In [51]:
class Gene_expression(nn.Module):
    def __init__(self):
        super(Gene_expression, self).__init__()
        self.b = nn.Parameter(torch.tensor(1.0),requires_grad=True)
        #self.sparsemax = Sparsemax(dim=-1) 
        self.softmax = nn.Softmax(dim=-1)

    def forward(self, x):

        b = self.b
        x = self.softmax(torch.exp(b) * x)
        return x

In [52]:
model = Gene_expression()
input_tensor = gene_expressions[0]
output = model(input_tensor)
print(output)

tensor([[2.8683e-01, 1.4554e-01, 2.3299e-01,  ..., 7.5374e-06, 3.7519e-06,
         8.7540e-06],
        [4.3083e-01, 1.5340e-01, 1.1692e-01,  ..., 5.4419e-06, 5.4419e-06,
         5.4419e-06],
        [6.0959e-01, 9.6138e-02, 9.5547e-02,  ..., 3.2407e-06, 3.2407e-06,
         1.2263e-06],
        [3.9632e-01, 1.2498e-01, 3.6335e-01,  ..., 1.0194e-06, 5.5423e-06,
         3.7666e-06],
        [4.7423e-01, 3.0722e-01, 1.2700e-01,  ..., 9.6103e-07, 3.8040e-07,
         6.2887e-07]], grad_fn=<SoftmaxBackward0>)


In [53]:
df = pd.read_csv('data/tumor_antigens.csv')

In [65]:
file_path = './data/tumor_antigens_fake.csv'
df = pd.read_csv(file_path)

# Extract gene names and HLA binding matrix
all_genes = df['Gene'].tolist()  # Gene names
hla_binder = torch.tensor(df.drop('Gene', axis=1).values, dtype=torch.float32)

In [66]:
hla_binder

tensor([[80., 12., 62.,  ..., 59., 39., 21.],
        [77., 86., 26.,  ..., 91., 49.,  7.],
        [ 2., 65.,  5.,  ..., 45., 40., 30.],
        ...,
        [66., 84., 87.,  ..., 11., 72., 16.],
        [79., 44., 73.,  ..., 32., 20., 56.],
        [26., 86., 82.,  ..., 99., 75., 61.]])

In [67]:
class Immunogenicity(nn.Module):
    def __init__(self, all_genes, hla_binder):
        super(Immunogenicity, self).__init__()
        self.all_genes = all_genes
        self.hla_binder = hla_binder  # HLA binding matrix (genes x HLA)
        
        # Learnable vector of length corresponding to the number of HLA types
        self.hla = nn.Parameter(torch.full((self.hla_binder.shape[1],), -1.0), requires_grad=True)
        
        # Map genes to indices for quick lookup
        self.gene_to_index = {gene: idx for idx, gene in enumerate(self.all_genes)}

    def forward(self, current_genes):
        # Filter genes to include only those present in all_genes
        filtered_genes = [gene for gene in current_genes if gene in self.gene_to_index]
        indices = [self.gene_to_index[gene] for gene in filtered_genes]
        
        if len(indices) == 0:
            return torch.zeros(0), []  # Return empty if no matching genes
        
        # Extract the HLA binding counts for the filtered genes
        print(indices)
        gene_binding_matrix = self.hla_binder[indices]  # Shape: (len(filtered_genes), num_HLA)
        print(gene_binding_matrix)
        # Perform element-wise multiplication with the learnable HLA vector
        hla_interaction = gene_binding_matrix * torch.exp(self.hla)  # Shape: (len(filtered_genes), num_HLA)
        
        # Sum over the HLA dimension to get the immunogenicity for each gene
        ig = torch.sum(hla_interaction, dim=1)  # Shape: (len(filtered_genes),)
        
        return torch.sigmoid(ig), filtered_genes


In [68]:
model = Immunogenicity(all_genes, hla_binder)
output, filtered_genes = model(list(current_genes[0]))
output,filtered_genes

[1493, 1529, 1531, 880, 2292, 1530, 2631, 1498, 1491, 1325]
tensor([[17., 21., 62., 75., 61., 79., 70., 90., 95.,  4., 19., 31., 59., 65.,
         24., 83., 16., 98., 40., 50., 18., 41., 92., 80., 33., 85., 37., 21.,
          3., 54., 29., 57.,  7., 82., 10.,  5., 56., 81., 20., 28., 54., 14.,
         60., 69.,  6., 24., 47., 27., 23., 29., 59., 82., 72., 89., 32., 41.,
         24., 56., 31., 10., 11., 22., 51., 89., 97.,  9., 43.,  5., 13., 51.,
         86.,  5., 50., 67., 29., 72., 13., 98., 72., 88., 69., 24., 27., 68.,
         33., 79.,  2., 80., 15., 36.],
        [70., 92., 52., 50., 58., 14., 98., 90., 24., 78., 79., 88., 73., 10.,
         42., 10.,  1., 53., 55., 56., 98., 45., 80., 77., 35., 20., 58., 29.,
         46., 92., 84., 50., 23., 15., 72., 58., 15.,  3., 71., 81., 88., 71.,
         21., 30., 62., 62.,  5., 46., 19., 41., 23., 48.,  1., 96., 64., 62.,
         62., 30., 16., 84., 67., 42., 11., 14., 94., 95., 71., 54., 17., 96.,
          9., 27., 64., 89., 30

(tensor([1., 1., 1., 1., 1., 1., 1., 1., 1., 1.], grad_fn=<SigmoidBackward0>),
 ['HIST1H1E',
  'HIST1H4C',
  'HIST1H4E',
  'CXCL9',
  'MMP9',
  'HIST1H4D',
  'OXTR',
  'HIST1H2AG',
  'HIST1H1B',
  'GBP5'])

In [58]:
len(filtered_genes)

473

In [59]:

class MIL(nn.Module):
    def __init__(self, all_genes, hla_binder):
        super(MIL, self).__init__()
        self.distance = Distance()
        self.gene_expression = Gene_expression()
        self.immunogenicity = Immunogenicity(all_genes, hla_binder)
        self.alpha = nn.Parameter(torch.tensor(1.0), requires_grad=True)
        self.beta = nn.Parameter(torch.tensor(1.0), requires_grad=True)
    
    def forward(self, distances, gene_expressions, current_genes):
        bag_outputs = []
        for distance, gene_expression in zip(distances, gene_expressions):
            distance = self.distance(distance)
            gene_expression = self.gene_expression(gene_expression)
            immunogenicity, filtered_genes = self.immunogenicity(current_genes)
            alpha = self.alpha
            beta = self.beta
        
            if len(filtered_genes) == 0:
                continue  # Skip if no overlapping genes
        
        # Print debug information
            #print(f"gene_expression shape: {gene_expression.shape}")
            #print(f"current_genes length: {len(current_genes)}")
            #print(f"filtered_genes length: {len(filtered_genes)}")
        
        
            gene_to_index = {gene: idx for idx, gene in enumerate(current_genes)}
        
            gene_indices = [gene_to_index[gene] for gene in filtered_genes if gene in gene_to_index]
            gene_expression = gene_expression[:, gene_indices]
        
            #print(f"Filtered gene_expression shape: {gene_expression.shape}")
            #print(f"Immunogenicity shape: {immunogenicity.shape}")
        
            z = gene_expression @ immunogenicity
            #print(f"z shape: {z.shape}")
            z = z.unsqueeze(1)
            #print(f"z shape: {z.shape}")
            #print(f"distance shape: {distance}")
            bag_output = distance * z
            bag_output = torch.sum(bag_output, dim=0)
            bag_output = torch.exp(alpha) * bag_output + beta
            bag_output = torch.sigmoid(bag_output)
            #print(bag_output)
            bag_outputs.append(bag_output)
            #df = pd.DataFrame(bag_outputs)
            #df.to_csv('output.csv')
    
        
        return torch.stack(bag_outputs).squeeze(dim=1)
    

class EarlyStopping:
    def __init__(self, patience=5, delta=0.001):
        self.patience = patience
        self.delta = delta
        self.best_loss = np.inf
        self.counter = 0
        self.stopped_epoch = 0
        self.early_stop = False

    def __call__(self, val_loss, model, epoch):
        if val_loss < self.best_loss - self.delta:
            self.best_loss = val_loss
            self.counter = 0
            torch.save(model.state_dict(), 'final_model.pth')
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True
                self.stopped_epoch = epoch

In [60]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')   

In [61]:
model = MIL(all_genes,hla_binder).to(device)
criterion = nn.BCELoss().to(device)
optimizer = torch.optim.SGD(model.parameters(), lr=0.001)
num_epochs = 100
patience = 5
early_stopping = EarlyStopping(patience=patience, delta=0.1)

In [62]:
for epoch in range(num_epochs):
    model.train() 
    running_loss = 0.0
    
    with tqdm(dataloader, unit="batch") as tepoch:
        for i, (distances, gene_expressions, label, core_idx, current_genes) in enumerate(tepoch):
            tepoch.set_description(f"Epoch {epoch+1}/{num_epochs}")

            optimizer.zero_grad()

            distances = torch.stack(distances).to(device)
            gene_expressions = torch.stack(gene_expressions).to(device)
            label = label.clone().detach().float().to(device)
            
            output = model(distances, gene_expressions, list(current_genes[0]))
            
            loss = criterion(output, label)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            tepoch.set_postfix(loss=loss.item())

    epoch_loss = running_loss / len(dataloader)
    print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {epoch_loss:.4f}')
    

    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for val_distances, val_gene_expressions, val_label, _, val_current_genes in val_dataloader:
            val_distances = torch.stack(val_distances).to(device)
            val_gene_expressions = torch.stack(val_gene_expressions).to(device)
            val_label = val_label.clone().detach().float().to(device)
            val_output = model(val_distances, val_gene_expressions, list(val_current_genes[0]))
            val_loss += criterion(val_output, val_label).item()
    
    val_loss /= len(val_dataloader)
    print(f'Validation Loss: {val_loss:.4f}')

    """early_stopping(val_loss, model, epoch)
    if early_stopping.early_stop:
        print(f'Early stopping at epoch {epoch+1}')
        break"""
        

Epoch 1/100: 100%|██████████| 858/858 [00:02<00:00, 401.29batch/s, loss=0.27]  


Epoch [1/100], Loss: 1.0086
Validation Loss: 0.9691


Epoch 2/100: 100%|██████████| 858/858 [00:02<00:00, 421.09batch/s, loss=1.82]  


Epoch [2/100], Loss: 0.8803
Validation Loss: 0.8759


Epoch 3/100: 100%|██████████| 858/858 [00:02<00:00, 393.26batch/s, loss=1.07] 


...

Epoch [7/100], Loss: 0.7400
Validation Loss: 0.7503


Epoch 8/100: 100%|██████████| 858/858 [00:01<00:00, 460.30batch/s, loss=0.741]


Epoch [8/100], Loss: 0.7341
Validation Loss: 0.7440


Epoch 9/100:  44%|████▍     | 380/858 [00:00<00:01, 458.33batch/s, loss=0.665]


KeyboardInterrupt: 